# Solar filament instance segmentation: demo

The Kaggle competition [filament-segmentation-2026](https://www.kaggle.com/competitions/filament-segmentation-2026)
asks for an instance mask of every solar filament in 2048 x 2048 GONG H-alpha full-disk images. A submission is a CSV
of COCO run-length-encoded masks, and it is scored by panoptic quality (PQ) pooled over every annotator's drawing of
every test image, where a prediction matches a ground-truth object when their IoU is above 0.5. This pipeline proposes
candidate masks from three U-Net probability maps, a Mask R-CNN detector and the I6 detector-refiner, predicts the
expected IoU of each candidate with LightGBM and ExtraTrees models, picks a non-overlapping set per image with an
integer program, and writes the masks. This notebook runs it stage by stage and rebuilds the evaluated submission
`s1-S-t024.csv` (Kaggle public score 0.41).

| # | stage | module | device |
|---|---|---|---|
| 1 | read the test images | `filseg/data.py` | CPU |
| 2 | U-Net probability maps (cnx, cnxL, R34; 4 fold models each) | `filseg/dense.py` | GPU (CPU possible) |
| 3 | Mask R-CNN candidates (4 fold models) | `filseg/det.py` | GPU (CPU possible) |
| 4 | I6 detector + refiner candidates | `filseg/i6.py` | GPU (CPU possible) |
| 5 | candidate masks and their 86 scorer columns | `filseg/evidence.py`, `filseg/rows.py` | CPU |
| 6 | expected IoU q of every candidate | `filseg/score.py` | CPU |
| 7 | exact selection of a conflict-free set | `filseg/select.py` | CPU |
| 8 | 0.23 px shift and pixel claiming | `filseg/shift.py` | CPU |
| 9 | the CSV | `filseg/write.py` | CPU |
| 10 | submission checks | `filseg/validate.py` | CPU |

Each stage below is one call to `filseg/pipeline.py`. `pipeline.run(mode=..., device=...)` runs the same chain in one
call.

## Setup

- `DATA`: the unzipped competition folder, the one that holds `test/test_images` (and `train/` for the evaluation
  section). `WORK`: a writable folder for intermediate files (several GB in full mode).
- `MODE = 'replay'` skips stages 1-4: it unpacks `replay/`, the stored stage 2-4 outputs of the run that produced the
  submission (every file is sha256-checked), and runs stages 5-10 on them. The result is byte-identical to the
  submitted file. It runs on the CPU and took about 21 minutes on a desktop CPU with 4 worker processes.
- `MODE = 'full'` runs every stage from the test images. Stages 2-4 run on `DEVICE`; on one RTX 3070 Ti the whole run
  took about 75 minutes. GPUs differ slightly in floating-point arithmetic and the U-Nets and I6 run in bfloat16, so
  the result is a close copy of the submission, not the same bytes. `DEVICE = 'cpu'` works but stages 2-4 then take
  many hours.
- `TRAIN = True` enables the optional retraining section at the end. It is off by default: the pipeline uses the
  trained weights shipped in `weights/`.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path.cwd()                      # run the notebook from the repository root
DATA = Path(os.environ.get('FILSEG_DATA', REPO / 'data' / 'MAGFiLO_1.0_Kaggle_2026'))
WORK = Path(os.environ.get('FILSEG_WORK', REPO / 'work'))

MODE = 'replay'      # 'replay': stored stage 2-4 outputs, exact rebuild on the CPU | 'full': every stage from the images
DEVICE = 'cuda'      # stages 2-4 in full mode (and the evaluation section): 'cuda' or 'cpu'
TRAIN = False        # optional retraining section at the end
NPROC = 4            # worker processes of stage 5

if DEVICE == 'cpu':
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'   # hide the GPUs before torch is imported ('' does not on Windows)
os.environ['FILSEG_DATA'], os.environ['FILSEG_WORK'] = str(DATA), str(WORK)
sys.path.insert(0, str(REPO))

from filseg import pipeline

IMAGES = pipeline.test_images(DATA)
WORK_RUN = WORK / f'run-{MODE}'        # one work folder per mode
OUT_CSV = WORK_RUN / 'submission.csv'
assert IMAGES.is_dir(), f'test images not found: {IMAGES}'
print(f'mode {MODE} | device {DEVICE} | {len(os.listdir(IMAGES))} test images in {IMAGES} | work {WORK_RUN}')

## Stage 1: test images (full mode)

Reads the 180 test JPEGs in sorted file-name order as 2048 x 2048 grayscale arrays, and computes for each image the
mean and standard deviation of the solar-disk pixels (those above 30 % of the 99th percentile). The U-Nets normalise
their input with these two numbers. The arrays are cached in `WORK_RUN/cache`.

In [ ]:
if MODE == 'full':
    recs, images = pipeline.stage1_images(IMAGES, WORK_RUN)
else:
    print('replay mode: stages 1-4 come from replay/ (cell after stage 4)')

## Stage 2: U-Net probability maps (full mode)

Three U-Nets with two output channels (filament mask, filament spine): `cnx` (ConvNeXt-T encoder, 24,000 training
steps), `cnxL` (the same network trained for 48,000 steps) and `R34` (ResNet34 encoder). Each has four fold models,
trained on the development folds other than its own. Every image goes through each model whole, in the 8 rotations
and flips of the square; the sigmoids are averaged over the views and the four fold models and stored as uint8.

What it contributes: the cnx mask map is the base of the candidate set (stage 5 thresholds it), the cnxL and R34 maps
add their own threshold candidates and the cross-map agreement columns of the scorer, and stage 8 reshapes the
selected masks on these maps.

In [ ]:
if MODE == 'full':
    maps = pipeline.stage2_dense(recs, images, WORK_RUN, DEVICE)

## Stage 3: Mask R-CNN candidates (full mode)

A torchvision Mask R-CNN (ResNet50-FPN v2) with four fold models, run at the native 2048 px resolution in float32.
Each fold model writes up to 100 detections per image (score above 0.03) with their masks, scores and mean mask
probabilities.

What it contributes: the detector's masks are not submitted directly. Stage 5 measures how each threshold candidate
overlaps the detections and turns that into scorer columns (best IoU, score of the overlapping detection, and so
on). The four fold models give four versions of these columns, and stage 6 averages the scorer over them.

In [ ]:
if MODE == 'full':
    det_roots = pipeline.stage3_det(recs, images, WORK_RUN, DEVICE)

## Stage 4: I6 detector and refiner candidates (full mode)

The I6 model from the project's earlier Codex track: a Faster R-CNN detector proposes boxes on nine overlapping
1024 px tiles, global NMS keeps at most 128 boxes, and a ResNet18 U-Net refines a crop around each box (resized to
512 x 512) into a mask. Its weights were fit on all
707 training images.

What it contributes: like stage 3, a second independent set of candidates whose overlap with the threshold
candidates becomes scorer columns.

In [ ]:
if MODE == 'full':
    i6_root = pipeline.stage4_i6(IMAGES, WORK_RUN, DEVICE, names=[r['file_name'] for r in recs])
    del images

## Replay: the stored outputs of stages 2-4 (replay mode)

Unpacks `replay/` into `WORK_RUN/replay`: the three map files, the four detector roots and the I6 bank of the run
that produced the submission. Every file is checked against the sha256 of the original.

In [ ]:
if MODE == 'replay':
    maps, det_roots, i6_root = pipeline.stage_replay(WORK_RUN)
print(maps)

## Stage 5a: tree candidates and their evidence

The cnx mask map is cut at 16 levels, t = 0.10, 0.15, ..., 0.85. Every 8-connected component of at least 40 px at a
level is a candidate; components nest across levels and form a tree. Each candidate gets 28 features (shape, mask
and spine map values, image contrast, position on the disk, place in the tree), and its overlaps with the I6 and Mask R-CNN candidates are measured (`ext/`, `feat/`).

In [ ]:
counts = pipeline.stage5a_evidence(maps, det_roots, i6_root, WORK_RUN, IMAGES, NPROC)

## Stage 5b: the candidate pool

Adds candidates that a single threshold cannot give: closings that join two or more tree nodes of a level, pairwise
merges of nearby nodes along a straight bridge, and the threshold trees of the cnxL and R34 maps. New candidates that
duplicate an earlier one (IoU >= 0.9) are dropped. Every candidate gets its 86 scorer columns, and the pairs of
candidates that overlap are listed as conflicts. On the submitted run this gives 47,101 candidates on 180 images.

In [ ]:
rows_dir = pipeline.stage5b_rows(maps, WORK_RUN, IMAGES, NPROC)

## Stage 6: scoring

Each candidate's expected IoU with a ground-truth filament, averaged over annotators:
q = 0.7 x (mean of 10 LightGBM models) + 0.3 x (mean of 10 ExtraTrees models), each also averaged over the four
detector versions of the columns. The LightGBM models were trained on the candidates of all 707 labelled images, the
ExtraTrees models on the 566 development images.

In [ ]:
scores = pipeline.stage6_score(rows_dir, WORK_RUN, pipeline.map_files(maps))

## Stage 7: selection

Each candidate gets s = q - 0.2 |t - 0.475| - 0.03 [not a cnx tree node], t its threshold level. Per image, the
selection keeps the set of candidates with no two overlapping that maximises the sum of (s - 0.24). Candidates with
s <= 0.24 are dropped first; each remaining connected group of conflicts is solved exactly as a 0/1 integer program
(HiGHS through `scipy.optimize.milp`). The threshold 0.24 sets how high the score of a candidate must be for it to
be kept.

In [ ]:
selection = pipeline.stage7_select(scores, WORK_RUN)

## Stage 8: 0.23 px shift and claiming

Each selected tree node (cnx, cnxL or R34) is re-cut from its own probability map after the map is shifted 0.23 px
toward row 0 by linear interpolation, keeping only pixels inside a 3 x 3 dilation of the original mask. Merged
candidates keep their masks. The selected masks then claim pixels in descending s, so the instances stay disjoint.

In [ ]:
final = pipeline.stage8_shift(rows_dir, scores, selection, maps)

## Stage 9: the CSV

Writes one row per instance, `<image stem>_<n>,<COCO RLE counts>`, with column-major RLE, images in sorted order and
no row for an image without instances. The writer reads the file back and checks that it gives exactly the intended
masks.

In [ ]:
res = pipeline.stage9_write(final, OUT_CSV)

## Stage 10: validation

Checks that do not rely on the writer's own read-back: header bytes and line ends, unique and sequential ids, every
instance non-empty, instances pairwise disjoint, RLE round trip, and the column-major orientation with a decoder that
does not use pycocotools (a transposed file is valid RLE, so only an independent decoder can catch it).

In [ ]:
ok = pipeline.stage10_validate(OUT_CSV, final)
same = res['sha256'] == pipeline.SUBMITTED_SHA256
print(f"sha256 {res['sha256']}: " + ('identical to the submitted s1-S-t024.csv' if same else
      'differs from the submitted s1-S-t024.csv (expected in full mode; see the comparison below)'))
print('stage times (s):', {k: round(v) for k, v in pipeline.TIMES.items()})

## Two test images and their predicted instances

Left: the image with the predicted instances in colour. Middle and right: a 512 px window around the largest
instance, with the instances and with the cnx probability map of stage 2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from filseg import data as fdata

files = final['files']
n_inst = [len(final['masks'][f]) for f in files]
show = [files[int(np.argmax(n_inst))], files[len(files) // 2]]
cnx = np.load(maps['cnx'], mmap_mode='r')
fig, axes = plt.subplots(len(show), 3, figsize=(16, 5.5 * len(show)))
for ax, f in zip(axes, show):
    j = files.index(f)
    img = fdata.read_image(IMAGES, f)
    lab = np.zeros(2048 * 2048, np.int32)
    for n, q in enumerate(final['masks'][f], 1):
        lab[q] = n
    lab = lab.reshape(2048, 2048)
    over = np.zeros((2048, 2048, 4), np.float32)
    over[lab > 0] = plt.cm.tab20((lab[lab > 0] - 1) % 20)
    over[..., 3] = 0.65 * (lab > 0)
    if final['masks'][f]:
        big = max(final['masks'][f], key=len)
        cy, cx = int(np.median(big // 2048)), int(np.median(big % 2048))
    else:
        cy = cx = 1024
    y0, x0 = min(max(cy - 256, 0), 1536), min(max(cx - 256, 0), 1536)
    win = (slice(y0, y0 + 512), slice(x0, x0 + 512))
    ax[0].imshow(img, cmap='gray'); ax[0].imshow(over)
    ax[0].add_patch(plt.Rectangle((x0, y0), 512, 512, fill=False, ec='w', lw=1))
    ax[0].set_title(f'{f}: {len(final["masks"][f])} instances')
    ax[1].imshow(img[win], cmap='gray'); ax[1].imshow(over[win]); ax[1].set_title('window: instances')
    ax[2].imshow(cnx[j, 0][win], cmap='magma', vmin=0, vmax=255); ax[2].set_title('window: cnx mask probability')
    for a in ax:
        a.axis('off')
plt.tight_layout()
plt.show()

## Comparison with the submitted file

`submission/s1-S-t024.csv` is the file that was scored on Kaggle. The comparison matches instances one-to-one per
image in descending IoU (pairs with IoU > 0.5, as in the metric) and counts pixel-identical and changed instances. In
replay mode the files are byte-identical. In full mode, on an RTX 3070 Ti, 1,203 of the 1,206 submitted instances were
matched, 910 of them pixel-identical; 3 were missing and 5 were new.

In [ ]:
pipeline.compare(pipeline.SUBMITTED_CSV, OUT_CSV)

## Evaluation on a development fold

The test labels are hidden (Kaggle scores the test predictions), so a PQ can only be computed on the training images,
whose annotations are part of the competition data. This cell runs the full pipeline on the first `EVAL_N` images of
development fold `EVAL_FOLD` and scores them with `filseg.metrics`, a replica of the organisers' metric: per annotator
record, every (ground truth, prediction) pair with IoU > 0.5 counts, and PQ = sum of matched IoU / (TP + 0.5 FP +
0.5 FN), pooled over records.

This number is in-sample and optimistic, not an estimate of the test score: each U-Net and detector average includes
three fold models that were trained on these images, and the I6 model and the LightGBM scorer were trained on all
707 labelled images. It checks that the chain and the metric run end to end on labelled data. Fold 4 is the reserved
holdout and is refused. Stages 2-4 run on `DEVICE`; on the CPU the U-Nets alone take about 8 minutes per image,
so `EVAL_N` is small there.

In [ ]:
ANN = DATA / 'train' / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
TRAIN_IMAGES = DATA / 'train' / 'train_images'
RUN_EVAL = True
EVAL_FOLD = 0
EVAL_N = 8 if DEVICE == 'cuda' else 2

if RUN_EVAL and ANN.exists() and TRAIN_IMAGES.is_dir():
    names = pipeline.dev_fold_images(EVAL_FOLD, EVAL_N)
    ev_work = WORK / f'eval-f{EVAL_FOLD}-n{EVAL_N}'
    ev = pipeline.run('full', DEVICE, work=ev_work, image_dir=TRAIN_IMAGES, names=names,
                      out_csv=ev_work / 'dev.csv', nproc=NPROC)
    pq = pipeline.evaluate(ev['csv'], names, ANN)
else:
    print(f'skipped: RUN_EVAL is False or the training annotations are not under {DATA / "train"}')

## Optional: retraining

`filseg/train.py` calls the training entry points in `training/` with the recorded arguments in `configs/<name>.json`.
It is off by default (`TRAIN = False`).

- The 16 networks with a training entry point are the 12 U-Nets (`cnx-f0..3`, `cnxL-f0..3`, `r34-f0..3`) and the 4
  Mask R-CNN fold models (`det-f0..3`). Retraining all of them takes many GPU hours: the recorded times on one
  RTX 4070 SUPER add up to about 26-30 hours. Both trainers need CUDA, about 8.9 GB of disk for the training cache,
  and an internet connection for the pretrained starting weights (ImageNet encoders, COCO Mask R-CNN).
- The I6 model was trained in the Codex track; its training code is not part of this release.
- The LightGBM and ExtraTrees scorers retrain on the CPU in minutes (about 10 s per LightGBM model and 90 s for all 10
  ExtraTrees) with `train.fit_scorers()`, which reads the two training tables the shipped models were fit on from
  `training/scorer_data/` (see its README). Refit with 4 LightGBM threads, the LightGBM files are byte-identical to
  the shipped ones. Building such tables from scratch needs out-of-fold maps and candidates on the training images;
  that code is not part of this release.

Training is not bit-reproducible, so a retrained network differs from the shipped one. Its weights are written to
`WORK/runs/<name>/`; the pipeline above keeps using `weights/`, and the scorer was fitted to the outputs of the
shipped networks.

In [ ]:
from filseg import train

TRAIN_MODEL = 'r34-f0'     # one of train.NETWORKS
print(f'{len(train.NETWORKS)} networks; recorded GPU time for all: up to {train.estimated_hours():.0f} h, '
      f'for {TRAIN_MODEL}: up to {train.estimated_hours([TRAIN_MODEL]):.1f} h')
if TRAIN:
    if not (train.default_cache() / 'meta.json').exists():
        train.prepare_cache(data=DATA, workers=NPROC)
    weights_file = train.train_network(TRAIN_MODEL)
    print('written:', weights_file)
else:
    print('TRAIN is False: nothing trained; the pipeline uses the shipped weights in weights/')